# 공개 데이터 SFT 2단계 — QLoRA 학습

## 이 노트북이 하는 일
`mixed_sft.jsonl`(NuminaMath 20k + RFT 5.4k)로 모델을 학습시킵니다.

## RFT 학습 때와 달라진 점 4가지

| | RFT 때 | 이번 | 이유 |
|---|---|---|---|
| 에폭 | 2 | **1** | 지난번 2에폭째가 과적합이었음 |
| 최적점 저장 | ❌ | **✅** | eval loss 최저 시점을 자동 복원 |
| 배치 | 1 x 16 | **4 x 4** | T4 활용도를 올려 속도 개선 |
| 긴 샘플 | 잘라냄 | **버림** | 잘린 풀이는 `\boxed{}`가 없어 학습에 해로움 |

## 시간이 관건입니다
RFT 학습이 5,356샘플 x 2에폭에 3시간 55분(step당 21초)이었습니다.
그대로면 25,465샘플은 **9시간 20분** — 세션 12시간 제한에 너무 붙습니다.

→ `N_TRAIN`으로 샘플 수를 줄이고, 배치를 키워 속도를 올립니다.
→ **`[6]`에서 ETA를 먼저 확인**하고 진행 여부를 판단하세요.

## 실행 순서
`[1]` `[2]` → ⛔Restart → `[1]` → `[3]`~`[7]`


---
## [1] 설정 ▶️ 항상 실행

### ★ 맨 위 두 줄이 중요합니다
```python
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
```
T4가 2장이면 Trainer가 자동으로 **DataParallel**(두 GPU에 나눠 돌리기)을 켭니다.
그런데 **4비트 양자화 모델은 복제가 안 됩니다** — 양자화 상태가 특정 GPU에 묶여 있어서요.
지난번 여기서 `CUDA illegal memory access`로 죽었습니다.

GPU를 한 장만 보이게 만들어 DataParallel 자체를 차단합니다.
**환경변수는 CUDA 초기화 전에 설정돼야 하므로 반드시 맨 위**여야 합니다.

### ★ `N_TRAIN`
25,465개 전부 쓰면 9시간이 넘습니다. 15,000개면 **약 4~5시간**으로 예상됩니다.
`[6]`에서 실제 ETA를 보고 조정하세요.

### ★ `BATCH=4, GRAD_ACC=4`
실질 배치는 16으로 지난번과 같습니다. 다만 한 번에 4개를 처리하니 **GPU 활용도가 올라갑니다.**
`BATCH=1`은 T4를 절반도 못 씁니다. OOM이 나면 2로 낮추세요(`GRAD_ACC`는 8로).

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # ★ 4bit + DataParallel 충돌 방지. 반드시 맨 위

# ── 데이터 ────────────────────────────────────────────
N_TRAIN    = 15000     # mixed_sft.jsonl 25,465개 중 사용할 수
MAX_LEN    = 1024      # 초과 샘플은 자르지 않고 버림
EVAL_RATIO = 0.01

# ── LoRA (지난번과 동일) ───────────────────────────────
LORA_R     = 32
LORA_ALPHA = 64
LORA_DROP  = 0.05

# ── 학습 ──────────────────────────────────────────────
EPOCHS     = 1         # 지난번 2에폭째가 과적합
LR         = 1e-4
BATCH      = 4         # OOM이면 2로 (GRAD_ACC는 8로)
GRAD_ACC   = 4         # 실질 배치 = 16
SEED       = 42

MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
OUT_DIR    = "/kaggle/working/qwen25-3b-numina-lora"
SYSTEM = ("You are an expert competition mathematician. Solve the problem step by step, "
          "concisely. The final answer is ALWAYS a single integer. "
          "End your response with the final integer inside \\boxed{}.")
# ──────────────────────────────────────────────────────
print(f"N_TRAIN={N_TRAIN} | {EPOCHS}epoch | 실질배치={BATCH*GRAD_ACC} | 예상 step={N_TRAIN//(BATCH*GRAD_ACC)}")

---
## [2] 설치 ⏭️ 세션 안 껐으면 건너뛰기
---
## ⛔ 설치 후 Run → Restart Session → [1]부터
---

In [ ]:
!pip install -q -U peft bitsandbytes accelerate 2>&1 | tail -3

import torch, peft, bitsandbytes, transformers
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("보이는 GPU  :", torch.cuda.device_count(), "(1이어야 함)")

---
## [3] 데이터 로드 ▶️

`mixed_sft.jsonl`을 읽고 `N_TRAIN`개를 무작위 추출합니다.
NuminaMath와 RFT가 섞인 상태로 셔플돼 있으므로, 무작위로 뽑으면 **비율이 자연스럽게 유지**됩니다.

In [ ]:
import json, glob, random
from collections import Counter

paths = [p for p in glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
         if "mixed" in os.path.basename(p).lower()]
if not paths:
    paths = glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
print("찾은 jsonl:", paths)
assert paths, "mixed_sft.jsonl 을 못 찾았습니다. + Add Input 으로 Dataset을 추가하세요."

data = [json.loads(l) for l in open(paths[0], encoding="utf-8")]
print(f"\n원본 {len(data):,}개  {dict(Counter(r.get('src','?') for r in data))}")

random.Random(SEED).shuffle(data)
data = data[:N_TRAIN]
print(f"추출 {len(data):,}개  {dict(Counter(r.get('src','?') for r in data))}")

---
## [4] 토큰화 + 손실 마스킹 ▶️

### 손실 마스킹 (지난번과 동일)
프롬프트 구간의 라벨을 `-100`으로 채웁니다. PyTorch에서 "손실 계산 제외"를 뜻하는 값이에요.
모델이 배워야 할 건 **풀이 쓰는 법**이지 문제 지어내는 법이 아니니까요.

### ★ 이번에 바뀐 점: 긴 샘플은 자르지 않고 버립니다

지난번엔 `MAX_LEN` 초과분을 `[:MAX_LEN]`으로 잘랐습니다. 그런데 **잘린 풀이는 끝에 `\boxed{}`가 없습니다.**
그걸 학습하면 모델이 **답 없이 끝내는 법**을 배웁니다. 파싱 실패율이 올라가는 원인이 되죠.

NuminaMath 풀이는 RFT보다 길어서 초과 비율이 더 높을 수 있습니다. 그래서 **아예 제외**합니다.
버려지는 비율이 15%를 넘으면 `MAX_LEN`을 1280으로 올리는 걸 고려하세요(메모리·시간 증가).

In [ ]:
import torch, pandas as pd
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print("eos:", repr(tok.eos_token), "| pad:", repr(tok.pad_token))

def build(r):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user",   "content": r["question"]}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full   = tok.apply_chat_template(
        msgs + [{"role": "assistant", "content": r["solution"]}], tokenize=False)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tok(full,   add_special_tokens=False)["input_ids"]
    if len(f_ids) > MAX_LEN:
        return None                       # ★ 자르지 않고 버림
    labels = list(f_ids)
    for i in range(min(len(p_ids), len(labels))):
        labels[i] = -100
    return {"input_ids": f_ids, "labels": labels}

built = [(build(r), r.get("src", "?")) for r in data]
ds  = [b for b, s in built if b is not None]
drop = len(built) - len(ds)
print(f"\nMAX_LEN={MAX_LEN} 초과로 제외: {drop:,}개 ({drop/len(built):.1%})")
if drop / len(built) > 0.15:
    print("▲ 15% 초과 — MAX_LEN을 1280으로 올리는 걸 고려하세요")
print("남은 src 분포:", dict(Counter(s for b, s in built if b is not None)))

lens = pd.Series([len(e["input_ids"]) for e in ds])
print(f"\n토큰 길이  중앙값 {lens.median():.0f} / p95 {lens.quantile(.95):.0f}")

n_eval = max(60, int(len(ds) * EVAL_RATIO))
eval_ds, train_ds = ds[:n_eval], ds[n_eval:]
tot = lens.sum() * EPOCHS
print(f"\n학습 {len(train_ds):,} / 검증 {len(eval_ds):,}")
print(f"총 학습 토큰 {tot/1e6:.1f}M   (RFT 때 5.3M → {tot/5.3e6:.1f}배)")

---
## [5] 4비트 모델 + LoRA ▶️ (지난번과 동일)

확인할 것 두 가지:
- `trainable params` 약 **5,987만 개(1.90%)**
- `GPU 사용` **2.9GB대** ← 4비트 양자화 성공. 6GB대면 실패

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4는 bf16 불가
    bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map={"": 0}, trust_remote_code=True)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROP,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"]))
model.print_trainable_parameters()
print(f"\nGPU 사용: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## [6] 학습 ▶️ ★ 시작 5분 뒤 ETA를 반드시 확인

### 이번에 추가된 설정
```python
save_strategy="steps", save_steps=100,
load_best_model_at_end=True,
metric_for_best_model="eval_loss",
```
**eval loss가 가장 낮았던 시점의 모델을 학습 끝에 자동으로 복원**합니다.
지난번엔 이게 없어서 과적합된 최종 모델밖에 못 건졌습니다.

`save_total_limit=2`로 디스크에는 최적 체크포인트와 최신 것만 남깁니다.

### ★ ETA 판단 기준

| ETA | 조치 |
|---|---|
| 5시간 이하 | 그대로 진행 |
| 5~7시간 | 진행하되 세션 12시간 제한 계산 |
| **7시간 초과** | **중단 → `N_TRAIN`을 10000으로** |

### 진행 중 볼 것
- train loss가 **1.0 근처에서 시작**할 것으로 예상됩니다.
  RFT 때는 0.24였는데, 그건 자기가 쓴 풀이라 이미 알던 것이었기 때문입니다.
  **이번엔 낯선 데이터라 loss가 높게 시작하는 게 정상이고, 오히려 배울 게 많다는 신호입니다.**
- `nan`이 나오면 즉시 중단 → `LR=5e-5`로 재시도

In [ ]:
from transformers import Trainer, TrainingArguments

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        pad = m - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [tok.pad_token_id] * pad)
        out["labels"].append(b["labels"] + [-100] * pad)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * pad)
    return {k: torch.tensor(v) for k, v in out.items()}

args = TrainingArguments(
    output_dir="/kaggle/working/ckpt",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    fp16=True,                              # T4
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=25,
    eval_strategy="steps",   eval_steps=100,
    save_strategy="steps",   save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,            # ★ 최적 시점 자동 복원
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    per_device_eval_batch_size=BATCH,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=eval_ds,
                  data_collator=collate)
trainer.train()

---
## [7] 어댑터 저장 ▶️

`load_best_model_at_end=True` 덕분에 **eval loss가 최저였던 시점의 어댑터**가 저장됩니다.

### 저장 후
1. 아래 링크로 **zip 다운로드** (세션 끄기 전!)
2. **Create → New Dataset** 업로드 (이름: `qwen25-3b-numina-lora`)
3. A/B 검증 노트북에서 Input으로 추가 → `LORA_PATH`가 자동 탐색됨

In [ ]:
import shutil

print("최적 체크포인트:", trainer.state.best_model_checkpoint)
print("최저 eval loss :", trainer.state.best_metric)

model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f}  ({os.path.getsize(os.path.join(OUT_DIR,f))/1e6:.1f} MB)")

zp = shutil.make_archive("/kaggle/working/numina_lora", "zip", OUT_DIR)
print(f"\nzip: {zp} ({os.path.getsize(zp)/1e6:.1f} MB)")

# 디스크 정리 (체크포인트는 더 이상 불필요)
shutil.rmtree("/kaggle/working/ckpt", ignore_errors=True)

from IPython.display import FileLink
FileLink("numina_lora.zip")